<a href="https://colab.research.google.com/github/wenh08/Python_projects-/blob/main/HospitalInpatientDischarge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1. Which medical conditions account for the most hospital discharges?
#2. Which conditions have the highest average treatment costs?
#3. Which conditions are associated with the longest hospital stays?
#4. How does illness severity affect costs for patients within the same diagnosis group?

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import StringIO
import gc

from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# Setting quoting=3 (csv.QUOTE_NONE) to ignore unclosed quotation marks causing EOF errors
df_full = pd.read_csv("/content/Hospital_Inpatient_Discharges_2024_2026.csv",
                      nrows=500000,

                      low_memory=False)

# only selecting 400K rows due to cloud storage
sample_size = 400000
random_state = 1

df = df_full.sample(n=sample_size, random_state=random_state)

# deleting the full dataset to keep memory clean
del df_full
gc.collect()

# checking to ensure the data was loaded properly
df.shape

(400000, 33)

In [13]:
df.head(10)

,Health Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code,Gender,Race,Ethnicity,...,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
352806,New York City,New York,7002020.0,1453.0,MEMORIAL HOSPITAL FOR CANCER AND ALLIED DISEASES,50-69,125,M,White,Not Span/Hispanic,...,Extreme,Extreme,Medical,Blue Cross/Blue Shield,NaN,NaN,NaN,Y,449321.39,145177.97
417824,Hudson Valley,Westchester,5957001.0,1139.0,WESTCHESTER MEDICAL CENTER,50-69,125,M,White,Not Span/Hispanic,...,Extreme,Major,Surgical,Medicare,Private Health Insurance,NaN,NaN,N,1118813.45,192747.71
469847,Long Island,Suffolk,5157003.0,943.0,ST CATHERINE OF SIENA HOSPITAL,70 or Older,NaN,F,Other Race,Not Span/Hispanic,...,Moderate,Minor,Surgical,Medicare,Blue Cross/Blue Shield,Private Health Insurance,NaN,N,111718.78,21940.79
407746,New York City,New York,7002054.0,1464.0,NEW YORK-PRESBYTERIAN HOSPITAL - COLUMBIA PRES...,70 or Older,100,F,Other Race,Spanish/Hispanic,...,Moderate,Moderate,Medical,Medicaid,NaN,NaN,NaN,Y,109000.53,21478.63
469848,Long Island,Suffolk,5154000.0,924.0,SOUTH SHORE UNIVERSITY HOSPITAL,50-69,117,M,Other Race,Spanish/Hispanic,...,Minor,Minor,Surgical,Miscellaneous/Other,NaN,NaN,NaN,N,63682.26,24109.54
494526,New York City,Bronx,7000006.0,3058.0,MONTEFIORE MED CENTER - JACK D WEILER HOSP OF ...,70 or Older,104,F,Other Race,Spanish/Hispanic,...,Minor,Moderate,Medical,Medicare,Medicaid,NaN,NaN,Y,41245.13,7166.43
413659,New York City,Bronx,7000001.0,1178.0,BRONXCARE HOSPITAL CENTER,50-69,NaN,M,Black/African American,Not Span/Hispanic,...,Extreme,Extreme,Medical,Medicaid,Medicaid,NaN,NaN,N,41960.70,31440.30
243752,New York City,New York,7002024.0,1456.0,MOUNT SINAI HOSPITAL,0-17,112,M,White,Not Span/Hispanic,...,Minor,Minor,Medical,Private Health Insurance,NaN,NaN,3000,N,23243.00,1902.74
157151,New York City,New York,7002020.0,1453.0,MEMORIAL HOSPITAL FOR CANCER AND ALLIED DISEASES,50-69,104,F,Other Race,Not Span/Hispanic,...,Minor,Minor,Surgical,Private Health Insurance,NaN,NaN,NaN,N,104189.39,41552.78
302827,Central NY,Onondaga,3301003.0,630.0,ST. JOSEPH'S HOSPITAL HEALTH CENTER,50-69,136,M,White,Not Span/Hispanic,...,Major,Extreme,Surgical,Federal/State/Local/VA,NaN,NaN,NaN,N,110229.25,75709.84


In [14]:
df.tail(10)

,Health Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code,Gender,Race,Ethnicity,...,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
184144,Hudson Valley,Westchester,5957001.0,1139.0,WESTCHESTER MEDICAL CENTER,18-29,105,F,Other Race,Unknown,...,Major,Minor,Medical,Medicaid,Medicaid,NaN,NaN,Y,336424.38,54822.42
357777,Capital/Adirondacks,Albany,101004.0,5.0,ST. PETER'S HOSPITAL,50-69,121,M,White,Not Span/Hispanic,...,Major,Major,Medical,Private Health Insurance,NaN,NaN,NaN,Y,58331.08,23604.23
219579,Long Island,Nassau,2950002.0,528.0,NASSAU UNIVERSITY MEDICAL CENTER,30-49,115,F,Other Race,Spanish/Hispanic,...,Moderate,Minor,Medical,Medicaid,Self-Pay,NaN,NaN,N,17911.37,6442.74
76349,New York City,New York,7002053.0,1463.0,NYU LANGONE HOSPITALS,70 or Older,112,M,White,Not Span/Hispanic,...,Major,Moderate,Medical,Medicare,Medicaid,NaN,NaN,Y,92606.23,33722.32
149002,New York City,New York,7002032.0,1469.0,MOUNT SINAI MORNINGSIDE,70 or Older,100,F,Black/African American,Not Span/Hispanic,...,Moderate,Major,Medical,Medicare,NaN,NaN,NaN,Y,118016.23,23503.22
80835,New York City,Richmond,7004003.0,1740.0,STATEN ISLAND UNIVERSITY HOSP-NORTH,50-69,103,M,Other Race,Spanish/Hispanic,...,Extreme,Extreme,Medical,Medicare,NaN,NaN,NaN,Y,144956.89,43921.94
11059,Hudson Valley,Westchester,5907002.0,1098.0,ST. JOSEPH'S MEDICAL CENTER,50-69,107,M,Black/African American,Not Span/Hispanic,...,Minor,Minor,Medical,Medicare,Medicare,Medicaid,NaN,Y,82719.32,36958.87
483443,Central NY,Onondaga,3301008.0,636.0,CROUSE HOSPITAL,50-69,132,F,Black/African American,Not Span/Hispanic,...,Moderate,Moderate,Medical,Medicaid,NaN,NaN,NaN,Y,25992.65,10290.28
205727,Capital/Adirondacks,Albany,101000.0,1.0,ALBANY MEDICAL CENTER HOSPITAL,0-17,128,M,White,Not Span/Hispanic,...,Minor,Minor,Medical,"Managed Care, Unspecified",NaN,NaN,NaN,Y,18123.85,4483.43
15809,New York City,New York,7002001.0,1438.0,BELLEVUE HOSPITAL CENTER,30-49,100,M,Black/African American,Unknown,...,Moderate,Moderate,Medical,Medicaid,NaN,NaN,NaN,N,81604.10,38090.67


In [13]:
# checking for missing values, naming them as missing and removing columns that have no missing values
#missing > 0

missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing.to_frame(name= "Missing Values")

,Missing Values
Birth Weight,369565
Payment Typology 3,358336
Payment Typology 2,224148
CCSR Procedure Description,120363
CCSR Procedure Code,120363
Zip Code,19961
Health Service Area,1348
Permanent Facility Id,1348
Hospital County,1348
Operating Certificate Number,1348


In [13]:
df.dtypes

,0
Health Service Area,object
Hospital County,object
Operating Certificate Number,float64
Permanent Facility Id,float64
Facility Name,object
Age Group,object
Zip Code,object
Gender,object
Race,object
Ethnicity,object
